In [3]:
source setup.tcl
set year 2016
set day 25
aoc::get-puzzle $year $day 1
aoc::get-puzzle $year $day 2

set input [string trim [aoc::get-input $year $day]];
jupyter::display text/plain [string range $input 0 100]...;

namespace eval ops {
    proc cpy {arg1 target} {
        if {$arg1 in {a b c d}} {
            set ::state($target) $::state($arg1)
        } else {
            set ::state($target) $arg1
        }
    }
    proc out {arg1} {
        incr ::state(nout)
        if {$arg1 in {a b c d}} {
            set val $::state($arg1)
        } else {
            set val $arg1
        }

        # puts \t$val
        
        if {$val == 0 && $::state(out) == 1} {
            set ::state(out) 0
            return
        }
        if {$val == 1 && $::state(out) == 0} {
            set ::state(out) 1
            return
        }
        error "Not valid sequence"
    }
    proc jnz {arg1 arg2} {
        if {$arg1 in {a b c d}} {
            set val $::state($arg1)
        } else {
            set val $arg1
        }
        if {$arg2 in {a b c d}} {
            set val2 $::state($arg2)
        } else {
            set val2 $arg2
        }
        if {$val != 0} {
                ::incr ::state(pc) $val2
                # offset the incr of pc after every command
                incr ::state(pc) -1

        }
        
    }
    
    proc tgl {arg1} {
        set offset [expr {$::state(pc) + $::state($arg1)}]
        # puts tgl-$offset
        set oldcmd [lindex $::state(prog) $offset]
        switch [llength $oldcmd] {
            2 { 
                lassign $oldcmd op1 arg1
                if {$op1 eq "inc"} {
                    set op1 dec
                } else {
                    set op1 inc
                }
                lset ::state(prog) $offset [list $op1 $arg1]
            }
            3 { 
                lassign $oldcmd op1 arg1 arg2
                if {$op1 eq "jnz"} {
                    set op1 cpy
                } else {
                    set op1 jnz
                }
                lset ::state(prog) $offset [list $op1 $arg1 $arg2]
            }
        }

    }
    proc inc {reg} {
       ::incr ::state($reg)
    }
    proc dec {reg} {
        ::incr ::state($reg) -1
    }
}

proc run {input a b c d} {
    set ::state(pc) 0
    set ::state(prog) [split $input \n]
    set ::state(toggled) {}
    set ::state(a) $a
    set ::state(b) $b
    set ::state(c) $c
    set ::state(d) $d
    set ::state(out) 1
    set ::state(nout) 0
    set visited {}
    set ::state(max) [llength $::state(prog)]
    while {$::state(pc) < $::state(max)} {
        if {$::state(pc) ni $visited} {
            lappend visited $::state(pc)
            # puts $visited
        }
        # puts ========
        set pc $::state(pc)
        set prog $::state(prog)
        # peephole optimisation
        if {[lrange $prog $pc $pc+4] eq [split {inc a
dec c
jnz c -2
dec d
jnz d -5} \n]} {
            # puts "Adding c*d to a"
            set c $::state(c)
            set d $::state(d)
            incr ::state(a) [expr {$c*$d}]
            set ::state(c) 0
            set ::state(d) 0
            incr ::state(pc) 5
            # puts "current command [lindex $::state(prog) $::state(pc)]"
            continue
        }
        set cmd [lindex $::state(prog) $::state(pc)]
        # parray ::state
        # puts "cmd: $::state(pc)"
        namespace eval ops [lindex $::state(prog) $::state(pc)]
        incr ::state(pc)
        # assume if we have had 100 valid outputs, it's the right answer
        if {$::state(nout) > 100} { return done }
    }
    return $::state(a)
}

(cached)

--- Day 25: Clock Signal --- You open the door and find yourself on the roof. The city sprawls away from you for miles and miles. There's not much time now - it's already Christmas, but you're nowhere near the North Pole, much too far to deliver these stars to the sleigh in time. However, maybe the huge antenna up here can offer a solution. After all, the sleigh doesn't need the stars, exactly; it needs the timing data they provide, and you happen to have a massive signal generator right here. You connect the stars you have to your prototype computer, connect that to the antenna, and begin the transmission. Nothing happens. You call the service number printed on the side of the antenna and quickly explain the situation. "I'm not sure what kind of equipment you have connected over there," he says, "but you need a clock signal." You try to explain that this is a signal for a clock. "No, no, a clock signal - timing information so the antenna computer knows how to read the data you're sending it. An endless, alternating pattern of 0 , 1 , 0 , 1 , 0 , 1 , 0 , 1 , 0 , 1 ...." He trails off. You ask if the antenna can handle a clock signal at the frequency you would need to use for the data from the stars. "There's no way it can! The only antenna we've installed capable of that is on top of a top-secret Easter Bunny installation, and you're definitely not-" You hang up the phone. You've extracted the antenna's clock signal generation assembunny code (your puzzle input); it looks mostly compatible with code you worked on just recently . This antenna code, being a signal generator, uses one extra instruction: 
 out x transmits x (either an integer or the value of a register) as the next value for the clock signal. The code takes a value (via register a ) that describes the signal to generate, but you're not sure how it's used. You'll have to find the input to produce the right signal through experimentation. 
 What is the lowest positive integer that can be used to initialize register a and cause the code to output a clock signal of 0 , 1 , 0 , 1 ... repeating forever?

(cached)

--- Part Two --- The antenna is ready. Now, all you need is the fifty stars required to generate the signal for the sleigh, but you don't have enough. You look toward the sky in desperation... suddenly noticing that a lone star has been installed at the top of the antenna! Only 49 more to go.

(cached)

cpy a d
cpy 9 c
cpy 282 b
inc d
dec b
jnz b -2
dec c
jnz c -5
cpy d a
jnz 0 0
cpy a b
cpy 0 a
cpy 2 c...

In [5]:
set i 0
while {1} {
#     puts $i
    if {![catch {run $input $i 0 0 0}]} break
    incr i
}
set i


192